### Сбор данных (при помощи APIs)

### 1. Первичное обращение по пользовательскому запросу

In [ ]:
# !pip install requests, datetime

In [ ]:
import datetime, requests, json


# 1. Входящие данные для поиска публикаций

# Ключ API – вводит пользователь (https://dev.elsevier.com/)
key = '<API_KEY>' 

# Поисковый запрос – вводит пользователь 
search = '<QUERY>'

# Интервал поиска в годах (нижняя граница) – вводит пользователь 
year_aft = 2015

# Интервал поиска в годах (верхняя граница) – вводит пользователь 
year_bef = 2025

# Предметная область запроса – вводит пользователь 
subjarea = '<SUBJECT>'

# Необходимо ли наличие doi у публикации – вводит пользователь (список: 0 или 1)
need_doi = 1

# Необходим ли поиск по конкретной организации (id из системы Scopus)
af_id = ''


# 2. Процедура создания запроса и поиска

query = f'{search} and pubyear aft {year_aft - 1} and pubyear bef {year_bef + 1} and language(english) and subjarea({subjarea})'

if need_doi:
    query = query + ' and doi(10*)'

if af_id:
    query = query + f' and af-id({af_id})'

fullquery = r'https://api.elsevier.com/content/search/scopus?start=0&count=1&query='+str(query)+'&apiKey='+str(key)
response = requests.get(fullquery)

paper_count = int(response.json()['search-results']['opensearch:totalResults'])
print(f"Количество публикаций по запросу составляет {paper_count}", "", sep="\n")

if paper_count > 5000:
    print("WARNING: Количество найденных публикаций превышает 5000.", "WARNING: Scopus Preview позволяет получить не более 5000 публикаций – измените параметры запроса", "", sep='\n')

print(f"INFO: По данному API доступно еще {response.headers['X-RateLimit-Remaining']} поисковых запросов")
print(f"INFO: Сброс запросов по данному API произойдет {datetime.datetime.fromtimestamp(int(response.headers['X-RateLimit-Reset']))}")

### 2. Загрузка списка релевантных публикаций (без аннотации)

In [ ]:
#!pip install pybliometrics

In [ ]:
# 3. Получение DataFrame со списком публикаций

from pybliometrics.scopus import ScopusSearch  # Документация: https://pybliometrics.readthedocs.io
import pandas as pd, pybliometrics

pybliometrics.scopus.init(keys=[key])

if paper_count > 5000:
    print("WARNING: Количество найденных публикаций превышает 5000.", "WARNING: Scopus Preview позволяет получить не более 5000 публикаций – измените параметры запроса", "", sep='\n')
else:
    print("INFO: Начинается скачивание публикаций по пользовательскому запросу", "", sep="\n")
    df = pd.DataFrame(ScopusSearch(query, subscriber=False, verbose=True, refresh=True).results)

    print()
    display(df)

### 3. Загрузка аннотации и дополнительных переменных с OpenAlex и CrossRef

In [ ]:
# 4.0 Предварительная подгрузка пользовательских функций

import re, json, requests

def openalex_parser(s): 
    listnewcols = ['OpenAlex_id','OpenAlex_abstract', 'OpenAlex_topconcepts', 'OpenAlex_authornames']
    global email

    if isinstance(s['doi'], str):
        url = 'https://api.openalex.org/works/https://doi.org/' + str(s['doi']) + '?mailto:' + str(email)
        response = requests.get(url)

        if response.status_code != 404:
            rj = requests.get(url).json()          
            oa_id = rj['ids']['openalex']
            s['OpenAlex_id'] = oa_id
            authors = []
            
            abstract_list = []
            if rj['abstract_inverted_index'] is not None:
                for x in rj['abstract_inverted_index'].keys():abstract_list.append(x)
                abstract = " ".join(abstract_list)
            else:
                abstract = 'none'
            s['OpenAlex_abstract'] = abstract
            
            concepts_list = []
            if rj['concepts'] is not None:
                for x in rj['concepts']: float(x['score']) > 0.6 and concepts_list.append(x['display_name'])
                concepts = ",".join(concepts_list)
            else:
                concepts = 'none'
            s['OpenAlex_topconcepts'] = concepts

            authors=[]
            for x in rj['authorships']:
                if x['author']['display_name'] is not None:
                    authors.append(x['author']['display_name'])
            s['OpenAlex_authornames']= ','.join(authors) if authors else 'none'

        else:
            for x in listnewcols: s[x] = 'not_in_OpenAlex'                
    else:
        for x in listnewcols: s[x] = 'no_doi' 
    return s

def crossref_parser(s):
    listnewcols = ['CrossRef_abstract']
    global email

    if isinstance(s['doi'], str):
        url = r'https://api.crossref.org/works/'+str(s['doi'])+'?mailto:'+str(email)
        response = requests.get(url)

        if response.text != 'Resource not found.':
            try:
                rj = requests.get(url).json()          

                s['CrossRef_abstract'] = re.sub('<[^<]+?>', '', rj['message']['abstract']) if 'abstract' in rj['message'] else 'no_abstract'
                
            except Exception as e:
                for x in listnewcols: s[x] = 'error:' + str(e)
        else:
            for x in listnewcols: s[x] = 'not_in_CrossRef'                
    else:
        for x in listnewcols: s[x] = 'no_doi' 
    return s

In [ ]:
#!pip install re
#!pip install tqdm
#!pip install openpyxl

In [ ]:
# 4.1 Загрузка аннотаций из баз OpenAlex и CrossRef

from tqdm import tqdm
import pandas as pd

# Осуществляя парсинг с OpenAlex и CrossRef вы соглашаетесь с их пользовательским соглашением – вводит пользователь (нужно окно)
email = '<email>'

# Название файла в формата xlsx с полученными данными – вводит пользователь (нужно окно)
export_path = 'scopus_export.xlsx'

tqdm.pandas()

print("INFO: Начинается скачивание аннотаций с OpenAlex")
try:
    df = df.progress_apply(openalex_parser, axis=1)
except Exception as e:
    print ('error:',str(e))

print("", "INFO: Начинается скачивание аннотаций с CrossRef", sep="\n")
try:
    df=df.progress_apply(crossref_parser, axis=1) 
except Exception as e:
    print ('error:',str(e))

df.sort_index(axis=1, inplace=True)

df = df[['subtypeDescription', 'title', 'creator', 'publicationName', 'volume', 'pageRange', \
        'doi', 'openaccess', 'affiliation_country', 'affiliation_city', 'affilname', \
        'OpenAlex_topconcepts', 'OpenAlex_abstract', 'CrossRef_abstract']]

df['abstract'] = df['OpenAlex_abstract'].mask((df['OpenAlex_abstract'] == 'none') | (df['OpenAlex_abstract'] == 'not_in_OpenAlex '), df['CrossRef_abstract'])

print("", "SUCCESS: Загрузка аннотаций завершена", sep="\n")

In [ ]:
# 4.2 Сохранение полученных результатов

df_abstract = df[~df['OpenAlex_abstract'].isin(['no_doi', 'none', 'not_in_OpenAlex']) & ~df['CrossRef_abstract'].isin(['no_abstract ', 'no_doi', 'not_in_CrossRef', 'none'])]

print(f'Аннотации доступны только в {len(df_abstract)} публикациях из {len(df)} доступных', '', sep='\n')

# Необходимо ли сохранить даже те публикации, у которых нет аннотации – вводит пользователь (список: 0 или 1)
full_save = 1

if full_save:
    df.to_excel(export_path)
else:
    df_abstract.to_excel(export_path)

print("SUCCESS: Сохранение списка публикаций в формате xlsx выполнено")